# Parking Spot Segmentation with SAM3

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/parking_spot_sam3.ipynb)

This notebook demonstrates how to segment parking spots from aerial imagery using the Segment Anything Model 3 (SAM3) through [segment-geospatial](https://samgeo.gishub.org).

SAM3 accepts a text prompt, so no training or labeled data is required. It uses the same aerial image as the [Parking Spot Detection](parking_spot_detection.ipynb) example, which instead uses a supervised Mask R-CNN model. The two approaches answer different questions:

- The supervised detector delineates **every parking stall**, whether occupied or empty, because it was trained on labeled stalls.
- SAM3 segments whatever a text prompt describes. Prompting for `car` yields the **occupied** spots, while prompting for `parking lot` outlines the lot itself.

Text prompts work best for objects with a distinct visual identity. Cars are segmented crisply, whereas an empty stall, which is just asphalt between two painted lines, is harder to isolate from text alone.

## Install package

To use SAM3, ensure `segment-geospatial` and a recent version of `transformers` are installed. Uncomment the commands below if needed.

In [ ]:
# %pip install "segment-geospatial[samgeo3]" geoai-py

In [ ]:
# %pip install "transformers>=5.0"

## Import libraries

In [ ]:
import geopandas as gpd

import geoai
from samgeo import SamGeo3

## Download sample data

We will use the same aerial image as the [Parking Spot Detection](parking_spot_detection.ipynb) example, so the results of the two approaches can be compared directly. You can find more high-resolution images from [OpenAerialMap](https://openaerialmap.org).

In [ ]:
raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/parking_spots.tif"
)

In [ ]:
raster_path = geoai.download_file(raster_url)

## Visualize the image

In [ ]:
geoai.view_raster(raster_url)

## Request access to SAM3

SAM3 is a gated model on Hugging Face. To use it, request access by filling out the form at https://huggingface.co/facebook/sam3.

Once you have access, uncomment the code block below and run it to log in.

In [ ]:
# from huggingface_hub import login

# login()

## Initialize SAM3

When initializing SAM3, you can choose the backend from `meta` or `transformers`.

In [ ]:
sam3 = SamGeo3(
    backend="transformers", device=None, checkpoint_path=None, load_from_HF=True
)

## Set the image

You can set the image by either passing the image path or the image URL.

In [ ]:
sam3.set_image(raster_path)

## Segment occupied parking spots

Prompt the model with `car` to segment every vehicle in the image. Each detected car marks an occupied parking spot. The `min_size` parameter discards masks smaller than the given number of pixels.

In [ ]:
sam3.generate_masks(prompt="car", min_size=10)

Check how many objects were found and how confident the model is about them.

In [ ]:
print(f"Objects found: {len(sam3.masks)}")
print(f"Mean confidence: {sum(sam3.scores) / len(sam3.scores):.3f}")

## Show the results

Display each object with a random color, along with its bounding box and confidence score.

In [ ]:
sam3.show_anns()

![](https://github.com/user-attachments/assets/376e1d12-9d1d-42c3-9868-2b45d49647bf)

Display the masks alone, with one unique value per object.

In [ ]:
sam3.show_masks()

## Save the masks

Save the masks as a GeoTIFF that carries the georeference of the input image.

In [ ]:
sam3.save_masks("parking_sam3_masks.tif")

## Convert masks to vector

Convert the mask raster to polygons so the results can be analyzed and shared as a GeoJSON file.

In [ ]:
sam3.raster_to_vector("parking_sam3_masks.tif", "parking_sam3.geojson")

In [ ]:
gdf = gpd.read_file("parking_sam3.geojson")
print(f"Polygons: {len(gdf)}")

## Add geometric properties

Compute area, length, and shape metrics for each polygon.

In [ ]:
gdf = geoai.add_geometric_properties(gdf)
gdf.head()

Vectorizing a mask raster can leave a few slivers along object edges. Drop the smallest polygons to keep one polygon per car.

In [ ]:
gdf = gdf[gdf["area_m2"] >= 2]
print(f"Occupied parking spots: {len(gdf)}")
print(f"Median car footprint: {gdf['area_m2'].median():.1f} m2")
print(f"Median car length: {gdf['major_length_m'].median():.1f} m")

## Visualize results

Overlay the segmented cars on the source imagery, colored by footprint area.

In [ ]:
geoai.view_vector_interactive(gdf, column="area_m2", tiles=raster_url)

## Segment the parking lot

The same image can be segmented at a different scale simply by changing the prompt. Prompting for `parking lot` outlines the lot itself rather than the vehicles in it. Here `min_size` is much larger, since a lot covers far more pixels than a car.

In [ ]:
sam3.generate_masks(prompt="parking lot", min_size=1000)
print(f"Objects found: {len(sam3.masks)}")

In [ ]:
sam3.show_anns()

![](https://github.com/user-attachments/assets/09e370f4-8b3f-4fb0-9017-773477ed5ef3)

## Interactive segmentation

Enter a text prompt or draw a rectangle on the map, then click the **Segment** button to run segmentation interactively. Try prompts such as `car`, `tree`, `building`, or `parking lot` to see how the results change.

In [ ]:
sam3.show_map(height="700px", min_size=10)

## Notes on prompt choice

SAM3 needs no training data, which makes it a good fit for exploratory mapping and for classes that have no dedicated model. The trade-off is that results follow the prompt's visual meaning rather than a fixed schema.

On this image:

- `car` segments individual vehicles cleanly, giving the occupied spots.
- `parking lot` outlines the lot as a whole.
- Prompts for an individual empty stall, such as `parking spot` or `empty parking space`, tend to return the lot or an aisle instead of one stall, because an empty stall has little visual identity of its own.

When you need every stall delineated, including the empty ones, a model trained on labeled stalls is the better tool. See the [Parking Spot Detection](parking_spot_detection.ipynb) example, which uses a supervised Mask R-CNN model on this same image.

For very large rasters, SAM3 also provides `generate_masks_tiled()`, which runs inference over a sliding window at native resolution instead of resizing the whole image.